# Publication Metrics for New BBB Models
Figures for official and repeated-trial results, including radar/spider plots, ROC curves, confusion matrices, feature importance, and SHAP summaries where available.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print(f'SHAP not available; permutation feature-importance plots will still be created: {exc}')

sns.set_theme(context='paper', style='whitegrid', font_scale=1.35)
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 600,
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

RUNS = {
    'all_descriptors': {
        'label': 'All PaDEL/RDKit Descriptors',
        'model_dir': Path('../output/models/hypertuning_v2'),
        'official': 'results_8020_v2_official.csv',
        'trials': 'trials_results_3runs_8020_v2.csv',
        'summary': 'trials_summary_mean_std_8020_v2.csv',
        'predictions': 'predictions_all_models_8020_v2.csv',
        'features': 'feature_names_8020_v2.pkl',
        'data_source': 'split',
    },
    'bbb_relevant': {
        'label': 'BBB-Relevant PaDEL Descriptors',
        'model_dir': Path('../output/models/bbb_relevant_padel_v2'),
        'official': 'results_bbb_relevant_official.csv',
        'trials': 'trials_results_3runs_bbb_relevant.csv',
        'summary': 'trials_summary_mean_std_bbb_relevant.csv',
        'predictions': 'predictions_all_models_bbb_relevant.csv',
        'features': 'bbb_relevant_descriptor_names.pkl',
        'data_source': 'bbb_relevant',
    },
}
FIGURE_DIR = Path('../figures/new_models_publication')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_available_results():
    loaded = {}
    for run_key, cfg in RUNS.items():
        model_dir = cfg['model_dir']
        if not (model_dir / cfg['summary']).exists():
            print(f"Skipping {run_key}: missing {(model_dir / cfg['summary'])}")
            continue
        loaded[run_key] = {
            'cfg': cfg,
            'official': pd.read_csv(model_dir / cfg['official']) if (model_dir / cfg['official']).exists() else None,
            'trials': pd.read_csv(model_dir / cfg['trials']),
            'summary': pd.read_csv(model_dir / cfg['summary']),
            'predictions': pd.read_csv(model_dir / cfg['predictions']) if (model_dir / cfg['predictions']).exists() else None,
        }
    return loaded


def flatten_summary(summary):
    rows = []
    metric_names = sorted({c[:-5] for c in summary.columns if c.endswith('_mean')})
    for _, row in summary.iterrows():
        for metric in metric_names:
            rows.append({
                'model': row['model'],
                'metric': metric,
                'mean': row.get(f'{metric}_mean', np.nan),
                'std': row.get(f'{metric}_std', np.nan),
            })
    return pd.DataFrame(rows)


def savefig(fig, name):
    fig.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{name}.pdf', bbox_inches='tight')


def model_order(summary, metric='roc_auc_mean'):
    return summary.sort_values(metric, ascending=False)['model'].tolist()


## Load New Model Results

In [ ]:
results = load_available_results()
for run_key, bundle in results.items():
    print('\n' + '=' * 80)
    print(bundle['cfg']['label'])
    display(bundle['summary'].sort_values('roc_auc_mean', ascending=False))


## Mean/Std Metric Panels

In [ ]:
metric_panels = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision', 'mcc']
for run_key, bundle in results.items():
    summary = bundle['summary']
    order = model_order(summary)
    plot_df = summary.set_index('model').loc[order]
    fig, axes = plt.subplots(3, 3, figsize=(18, 14), constrained_layout=True)
    for ax, metric in zip(axes.ravel(), metric_panels):
        means = plot_df[f'{metric}_mean']
        stds = plot_df[f'{metric}_std'].fillna(0)
        ax.bar(order, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
        ax.set_title(metric.replace('_', ' ').title(), fontweight='bold')
        ax.set_ylabel('Mean ± SD')
        ax.tick_params(axis='x', rotation=35)
        if metric != 'mcc':
            ax.set_ylim(0, 1.02)
    fig.suptitle(f"{bundle['cfg']['label']}: Three-Trial Mean ± SD", fontsize=18, fontweight='bold')
    savefig(fig, f'{run_key}_metrics_mean_std_panel')
    plt.show()


## Spider/Radar Charts

In [ ]:
radar_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc']
for run_key, bundle in results.items():
    summary = bundle['summary']
    order = model_order(summary)[:6]
    angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(10, 10))
    ax = plt.subplot(111, polar=True)
    for model in order:
        row = summary[summary['model'] == model].iloc[0]
        values = [row[f'{m}_mean'] for m in radar_metrics]
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=model)
        ax.fill(angles, values, alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m.replace('_', ' ').title() for m in radar_metrics], fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_title(f"{bundle['cfg']['label']}\nSpider Chart of Mean Metrics", fontsize=17, fontweight='bold', pad=28)
    ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.12), frameon=True)
    savefig(fig, f'{run_key}_spider_radar_metrics')
    plt.show()


## ROC-AUC Trial Distributions

In [ ]:
for run_key, bundle in results.items():
    trials = bundle['trials']
    order = trials.groupby('model')['roc_auc'].mean().sort_values(ascending=False).index.tolist()
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.boxplot(data=trials, x='model', y='roc_auc', order=order, ax=ax, color='#A0CBE8', width=0.55)
    sns.stripplot(data=trials, x='model', y='roc_auc', order=order, ax=ax, color='black', size=5, jitter=0.12)
    ax.set_title(f"{bundle['cfg']['label']}: ROC-AUC Across Three Trials", fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel('ROC-AUC')
    ax.set_ylim(0, 1.02)
    savefig(fig, f'{run_key}_roc_auc_trial_distribution')
    plt.show()


## ROC Curves from Official 80/20 Test Set

In [ ]:
for run_key, bundle in results.items():
    pred = bundle['predictions']
    if pred is None:
        continue
    fig, ax = plt.subplots(figsize=(8, 7))
    for model, g in pred.groupby('model'):
        fpr, tpr, _ = roc_curve(g['y_true'], g['y_score'])
        ax.plot(fpr, tpr, linewidth=2, label=f'{model} (AUC={auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    ax.set_title(f"{bundle['cfg']['label']}: ROC Curves", fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', frameon=True)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    savefig(fig, f'{run_key}_roc_curves_official')
    plt.show()


## Individual Per-Model Figures


In [ ]:
individual_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision']

for run_key, bundle in results.items():
    summary = bundle['summary']
    pred = bundle['predictions']
    order = model_order(summary)

    for model in order:
        row = summary[summary['model'] == model].iloc[0]
        means = [row[f'{m}_mean'] for m in individual_metrics]
        stds = [row[f'{m}_std'] for m in individual_metrics]

        fig, ax = plt.subplots(figsize=(10, 5.8))
        labels = [m.replace('_', ' ').title() for m in individual_metrics]
        ax.bar(labels, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
        ax.set_title(f"{bundle['cfg']['label']}: {model} Mean Metrics ± SD", fontweight='bold')
        ax.set_ylabel('Mean ± SD')
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=35)
        savefig(fig, f'{run_key}_{model}_individual_metric_profile')
        plt.show()

        if pred is not None:
            g = pred[pred['model'] == model]

            fig, ax = plt.subplots(figsize=(7, 6))
            fpr, tpr, _ = roc_curve(g['y_true'], g['y_score'])
            ax.plot(fpr, tpr, linewidth=2.5, color='#E45756', label=f'AUC={auc(fpr, tpr):.3f}')
            ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
            ax.set_title(f"{bundle['cfg']['label']}: {model} ROC Curve", fontweight='bold')
            ax.set_xlabel('False Positive Rate')
            ax.set_ylabel('True Positive Rate')
            ax.legend(loc='lower right', frameon=True)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.02)
            savefig(fig, f'{run_key}_{model}_individual_roc_curve')
            plt.show()

            fig, ax = plt.subplots(figsize=(5.7, 5.2))
            cm = confusion_matrix(g['y_true'], g['y_pred'], labels=[0, 1])
            ConfusionMatrixDisplay(cm, display_labels=['BBB-', 'BBB+']).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
            ax.set_title(f"{bundle['cfg']['label']}: {model} Confusion Matrix", fontweight='bold')
            savefig(fig, f'{run_key}_{model}_individual_confusion_matrix')
            plt.show()


## Confusion Matrices

In [ ]:
for run_key, bundle in results.items():
    pred = bundle['predictions']
    if pred is None:
        continue
    models = pred['model'].unique().tolist()
    ncols = min(3, len(models))
    nrows = int(np.ceil(len(models) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, model in zip(axes, models):
        g = pred[pred['model'] == model]
        cm = confusion_matrix(g['y_true'], g['y_pred'], labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=['BBB-', 'BBB+']).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
        ax.set_title(model, fontweight='bold')
    for ax in axes[len(models):]:
        ax.axis('off')
    fig.suptitle(f"{bundle['cfg']['label']}: Official Test Confusion Matrices", fontsize=17, fontweight='bold')
    savefig(fig, f'{run_key}_confusion_matrices')
    plt.show()


## Feature Importance and SHAP Charts

In [ ]:
def load_feature_data(run_key, cfg):
    if cfg['data_source'] == 'split':
        X_train = pd.read_csv('../data/split/8020/x_train.csv', index_col=0).select_dtypes(include=[np.number])
        y_train = pd.read_csv('../data/split/8020/y_train.csv', index_col=0).iloc[:, 0].astype(int)
        X_test = pd.read_csv('../data/split/8020/x_test.csv', index_col=0)[X_train.columns].select_dtypes(include=[np.number])
        y_test = pd.read_csv('../data/split/8020/y_test.csv', index_col=0).iloc[:, 0].astype(int)
        return X_train, y_train, X_test, y_test

    df = pd.read_csv('../data/padel_results_with_bbb.csv').dropna(subset=['BBB']).copy()
    features = joblib.load(cfg['model_dir'] / cfg['features'])
    X = df[features].select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    y = df['BBB'].astype(int)
    from sklearn.model_selection import StratifiedShuffleSplit
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr, te = next(split.split(X, y))
    return X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te]

for run_key, bundle in results.items():
    cfg = bundle['cfg']
    X_train, y_train, X_test, y_test = load_feature_data(run_key, cfg)
    top_models = model_order(bundle['summary'])

    for model_name in top_models:
        model_path = cfg['model_dir'] / f'{model_name}_8020_v2_best_model.pkl'
        if run_key == 'bbb_relevant':
            model_path = cfg['model_dir'] / f'{model_name}_bbb_relevant_best_model.pkl'
        if not model_path.exists():
            continue
        model = joblib.load(model_path)
        sample_X = X_test.sample(min(200, len(X_test)), random_state=42)
        sample_y = y_test.loc[sample_X.index]

        perm = permutation_importance(model, sample_X, sample_y, n_repeats=4, random_state=42, scoring='roc_auc', n_jobs=1)
        imp = pd.DataFrame({'feature': sample_X.columns, 'importance': perm.importances_mean, 'std': perm.importances_std})
        imp = imp.sort_values('importance', ascending=False).head(20).sort_values('importance')
        fig, ax = plt.subplots(figsize=(9, 8))
        ax.barh(imp['feature'], imp['importance'], xerr=imp['std'], color='#59A14F', edgecolor='black', linewidth=0.5)
        ax.set_title(f'{cfg["label"]}: {model_name} Permutation Importance', fontweight='bold')
        ax.set_xlabel('Mean ROC-AUC Decrease')
        ax.set_ylabel('')
        savefig(fig, f'{run_key}_{model_name}_permutation_importance')
        plt.show()

        if SHAP_AVAILABLE and model_name in model_order(bundle['summary'])[:3]:
            try:
                transformed = sample_X.copy()
                feature_names = np.array(sample_X.columns, dtype=object)
                for step_name, step in model.steps[:-1]:
                    if step == 'passthrough' or step_name == 'sampler':
                        continue
                    if hasattr(step, 'transform'):
                        transformed = step.transform(transformed)
                    if hasattr(step, 'get_feature_names_out'):
                        try:
                            feature_names = step.get_feature_names_out(feature_names)
                        except Exception:
                            pass
                clf = model.steps[-1][1]
                explainer = shap.Explainer(clf, transformed, feature_names=feature_names)
                shap_values = explainer(transformed[:200])
                plt.figure(figsize=(9, 7))
                shap.plots.beeswarm(shap_values, max_display=20, show=False)
                plt.title(f'{cfg["label"]}: {model_name} SHAP Summary', fontsize=15, fontweight='bold')
                plt.tight_layout()
                plt.savefig(FIGURE_DIR / f'{run_key}_{model_name}_shap_beeswarm.png', dpi=600, bbox_inches='tight')
                plt.savefig(FIGURE_DIR / f'{run_key}_{model_name}_shap_beeswarm.pdf', bbox_inches='tight')
                plt.show()
            except Exception as exc:
                print(f'SHAP skipped for {run_key}/{model_name}: {exc}')


## Export Compact Metric Table

In [ ]:
tables = []
for run_key, bundle in results.items():
    tmp = bundle['summary'].copy()
    tmp.insert(0, 'run', run_key)
    tables.append(tmp)
if tables:
    final_table = pd.concat(tables, ignore_index=True)
    final_table.to_csv(FIGURE_DIR / 'new_model_metrics_mean_std_table.csv', index=False)
    display(final_table)
print(f'Figures and tables saved to: {FIGURE_DIR}')
